# Testing AE convnext

In [ ]:
import torch
import torch.nn as nn
from models.blocks import ConvNeXtcausal
from torch.nn.utils.parametrizations import weight_norm
from transformers import EncodecModel
import sys
sys.path.append('../stable-audio-3')
from stable_audio_3 import AutoencoderModel

In [ ]:
class EncoderFast(nn.Module):
    def __init__(self, in_channels: int, dim: int, latent_dim: int, inter_channels: int, num_blocks: int):
        super(EncoderFast, self).__init__()

        stride = [4, 4, 4]
        self.conv1 = weight_norm(nn.Conv1d(in_channels, dim, kernel_size=7, padding=6))
        self.blocks = [ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)]
        self.stages= nn.ModuleList()

        for s in stride:
            stage = nn.Sequential(
                *self.blocks,
                nn.Conv1d(dim, dim, kernel_size=s, stride=s, padding=s-1)
            
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.proj = nn.Linear(dim, latent_dim)
    
    def forward(self, x):
        x = self.conv1(x)
        print(x.shape)
        for stage in self.stages:
            x = stage(x)
        print(x.shape)
        #x = x.mean(dim=-1)
        x = x.transpose(1,2)
        print(x.shape)
        x = self.norm(x)
        print(x.shape)
        x = self.proj(x)
        x = x.transpose(1,2)
        return x

In [ ]:
x = torch.randn(1, 1, 24000)
model = EncodecModel.from_pretrained('facebook/encodec_24khz')
with torch.no_grad():
    emb = model.encoder(x)
print(f"Encoder output shape: {emb.shape}")

In [ ]:
sample_rate = 24000
x = torch.randn(1, 1, sample_rate*1)
ae = AutoencoderModel.from_pretrained("same-s")
with torch.no_grad():
    emb = ae.encode(x, sample_rate)
print(f"Autoencoder output shape: {emb.shape}")

In [ ]:
class Decoder(nn.Module):
    def __init__(self, in_channels: int, dim: int, shift_dim: int, inter_channels: int, num_blocks: int):
        super(Decoder, self).__init__()
        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv = nn.Conv1d(in_channels, dim, kernel_size=7, padding=0)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.blocks = nn.ModuleList([ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)])
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, shift_dim, bias=False) 
        # (B, shift_dim, T) -> (B, 1 , shift_dim * T)
    
    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv(x)
        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.norm(x)
        x = x.transpose(1, 2)  # (B, dim, T)

        for block in self.blocks:
            x = block(x)

        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.linear1(x)
        x = self.linear2(x) # (B, T, shift_dim)
        x = x.view(x.size(0), -1) # (B, shift_dim * T)

        return x

In [ ]:
s_dim = x.size(-1) // emb.size(-1)
decoder = Decoder(in_channels=256, dim=512, shift_dim=s_dim, inter_channels=256, num_blocks=2)
decoder = decoder.to(emb.device) 
y = decoder(emb)
print(f"Decoder output shape: {y.shape}")